# Phase 1b — IDRiD lesion masks (top-up)

A small, self-contained top-up that caches **IDRiD Part A images together with their
lesion masks**. Nothing else is touched.

## Why this exists separately

The first Phase 1 run cached IDRiD images with no masks, because `find_mask()` only
looked for `<stem>.<ext>` while IDRiD names its masks with a channel suffix
(`IDRiD_55.jpg` → `IDRiD_55_MA.tif`). That is fixed, but re-running
`01_build_cache.ipynb` is the wrong way to apply it:

- `/kaggle/working` starts empty, so `build_cache.py` would rebuild all 5 GB from
  scratch rather than resuming.
- `verify-dr-cache-512` is linked to that notebook's output. A Kaggle dataset version
  is a **complete snapshot, not a diff** — publishing a version from an IDRiD-only run
  would replace the whole cache with just IDRiD.

So this notebook writes a *separate* small dataset. Phase 2 resolves the cache root
per dataset and prefers the copy with more mask channels, so the two are merged
automatically.

## Notebook settings

| Setting | Value |
|---|---|
| Accelerator | **None** — no GPU quota |
| Persistence | No persistence |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

## Inputs

**Raw IDRiD only.** Do not attach `verify-dr-cache-512` — it is not needed, and
leaving it off keeps this notebook's output small and unambiguous.

## Exit condition

Every lesion channel reports `written > 0`. The notebook asserts it, so a repeat of
the silent-absent failure cannot pass.

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"
print("repo ready at", REPO_DIR)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace("%20", "")

def dataset_roots(max_depth=3):
    """Candidate dataset directories, shallowest first.

    Kaggle does not always mount datasets as direct children of /kaggle/input --
    they can sit under competitions/ and datasets/ wrappers. Breadth-first so a
    shallower match always wins over a nested subfolder of the same name.
    """
    level, out = [INPUT], []
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                children = sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                continue
            out.extend(children)
            nxt.extend(children)
        level = nxt
    return out

EXCLUDE_ROOTS = []      # set to [CACHE] once the cache is located

def _under(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook.

    Anything under EXCLUDE_ROOTS is skipped. The cache contains directories named
    ddr, eyepacs, idrid, aptos and messidor2 -- exactly the keywords searched for --
    so without this a raw-dataset lookup can land inside the cache instead.
    """
    for d in dataset_roots():
        if any(_under(d, r) for r in EXCLUDE_ROOTS):
            continue
        if all(_norm(k) in _norm(d.name) for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print("     Use 'Add Input' in the right-hand panel. Candidate dirs seen:")
        for d in dataset_roots(2)[:25]:
            print(f"       - {d.relative_to(INPUT)}")
    return None

def find_any(*keyword_sets, label="", required=True):
    """Try several keyword spellings. Mirrors title themselves inconsistently:
    IDRiD ships as 'idrid-dataset' or 'indian-diabetic-retinopathy-image-dataset'."""
    for kws in keyword_sets:
        hit = find_mount(*kws, required=False)
        if hit:
            return hit
    if required:
        print(f"  !! NOT MOUNTED: {label or keyword_sets[0]}")
        print("     Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def match_channel(dirname):
    """Map a mask directory name to a lesion channel.

    DDR uses MA/HE/EX/SE; IDRiD uses '1. Microaneurysms', '2. Haemorrhages',
    '3. Hard Exudates', '4. Soft Exudates', '5. Optic Disc'. Match on meaning so
    one function covers both.
    """
    n = re.sub(r"^\d+\.\s*", "", dirname.lower().strip())
    n = n.replace("%20", " ")
    if n == "ma" or "microaneurysm" in n:
        return "microaneurysm"
    if n == "he" or "haemorrhage" in n or "hemorrhage" in n:
        return "haemorrhage"
    if n == "ex" or ("hard" in n and "exudate" in n):
        return "hard_exudate"
    if n == "se" or ("soft" in n and "exudate" in n) or "cotton" in n:
        return "soft_exudate"
    if n == "od" or "optic disc" in n:
        return "optic_disc"
    return None

MASK_EXT = {".tif", ".tiff", ".png", ".gif", ".bmp", ".jpg", ".jpeg"}
LESION4 = ["microaneurysm", "haemorrhage", "hard_exudate", "soft_exudate"]

def scan_mask_dirs(root, require=""):
    """Find mask directories under root, keyed by lesion channel."""
    found = {}
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if require and require not in str(d).lower():
            continue
        channel = match_channel(d.name)
        if not channel:
            continue
        files = [f for f in d.rglob("*") if f.suffix.lower() in MASK_EXT]
        if files:
            found.setdefault(channel, []).append((str(d), len(files)))
    return found

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

import shlex, subprocess

CACHE = Path("/kaggle/working/cache512")

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    print("$", cmd[:150] + (" ..." if len(cmd) > 150 else ""))
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(proc.stdout.rstrip())
    if proc.returncode != 0:
        print(proc.stderr.rstrip())
        raise RuntimeError(f"command failed with exit {proc.returncode}")
    return proc.stdout

## 2 · Find IDRiD Part A

Part A pairs original images with per-lesion mask folders. Both are discovered rather
than hardcoded, so the `%20` and spaces in IDRiD's directory names cause no trouble.

In [ ]:
idrid = find_any(("idrid",), ("indian", "diabetic", "retinopathy"), label="IDRiD")
assert idrid is not None, "raw IDRiD is not mounted - add it with 'Add Input'"

masks = {c: sorted(d for d, _ in v) for c, v in scan_mask_dirs(idrid).items()}
images = sorted(
    str(d) for d in idrid.rglob("*")
    if d.is_dir() and "original" in str(d).lower() and "segmentation" in str(d).lower()
    and any(f.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif"} for f in d.iterdir() if f.is_file()))

print("mask channels:")
for c in sorted(masks):
    print(f"   {c:15s} {len(masks[c])} dir(s)")
print("\nPart A image dirs:")
for d in images:
    print("   ", d)

assert masks, "no IDRiD mask directories found - is Part A present in this mirror?"
assert images, "no Part A original-image directories found"

## 3 · Cache images and masks together

One pass per split. The optic-disc channel is cached too — not a lesion, but it gives
M2b a stronger geometry signal than centre coordinates alone.

In [ ]:
def split_key(path):
    low = path.lower()
    return "train" if "train" in low else ("test" if "test" in low else None)

for img_dir in images:
    key = split_key(img_dir)
    chosen = {}
    for channel, dirs in masks.items():
        match = [d for d in dirs if split_key(d) == key]
        if match:
            chosen[channel] = match[0]
    if not chosen:
        print(f"skip {key}: no mask dirs for this split")
        continue
    print(f"\n=== IDRiD Part A ({key}): {len(chosen)} channels ===")
    cmd = (f"python {q(REPO_DIR / 'scripts/build_cache.py')} "
           f"--source-root {q(img_dir)} --dataset IDRiD --output-root {q(CACHE)} "
           f"--contact-sheet 40")
    for channel, d in chosen.items():
        cmd += f" --mask {q(f'{channel}={d}')}"
    run(cmd)

## 4 · The gate — were masks actually written?

This is the check the first run lacked. `written: 0, absent: N` means the directories
were found but no filename matched, which is how the masks went missing silently the
first time.

In [ ]:
report = json.loads((CACHE / "idrid" / "cache_report.json").read_text())
mask_root = CACHE / "idrid" / "masks"

print(f"cached images: {report.get('cached', 0)} / {report['counts']['found']}")
print(f"crop fallback: {report['crop']['fallback_rate']}")
print()
print(f"{'channel':<18}{'written':>9}{'absent':>9}{'failed':>9}{'files on disk':>15}")
print("-" * 60)
# Gate on files on disk, not on this run's counters: a resumed run legitimately
# writes nothing while every mask is already present.
ok = True
for channel, stat in sorted(report["masks"].items()):
    on_disk = len(list((mask_root / channel).glob("*"))) if (mask_root / channel).is_dir() else 0
    good = on_disk > 0 and stat["failed"] == 0
    if channel != "optic_disc":
        ok &= good
    print(f"{channel:<18}{stat['written']:>9}{stat['absent']:>9}{stat['failed']:>9}"
          f"{on_disk:>15}{'' if good else '   <-- NO MASKS ON DISK'}")

print()
assert ok, (
    "at least one lesion channel has no masks on disk. A non-zero 'absent' means the "
    "mask directory was found but no filename matched the image stem - check the "
    "naming convention before publishing this cache.")
print("PASS - IDRiD lesion masks cached. Publish and re-run Phase 2.")

## 5 · Eyeball the crops

In [ ]:
from IPython.display import Image, display
for sheet in sorted(CACHE.rglob("contact_sheet.jpg")):
    display(Image(str(sheet)))

---
## 6 · Publish

**Save Version → Save & Run All (Commit).** Then Output tab → **New Dataset**, named
**`verify-dr-idrid-masks`**.

> **Create a new dataset. Do not publish a new version of `verify-dr-cache-512`.**
> That dataset is linked to `01_build_cache`, and a version is a complete snapshot —
> a version made from this notebook's output would replace all 5 GB with IDRiD alone.

Then run `02_manifests.ipynb` with **both** cache datasets attached plus the raw
sources. It resolves the cache root per dataset and prefers the copy with more mask
channels, so IDRiD comes from here and everything else from the original cache. The
notebook prints the mapping when more than one root is in play.